In [2]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate flask pyngrok huggingface_hub

In [3]:
# Cell 2: Authenticate
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("NGROK_AUTH_TOKEN")

login(token=secret_value_0)
print("HF login OK")

HF login OK


In [4]:
# Cell 3: Load MedGemma model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/medgemma-4b-it"  # instruction-tuned variant

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=secret_value_0)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=secret_value_0,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"Model loaded on {model.device}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

Loading google/medgemma-4b-it...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

Model loaded on cuda:0
GPU memory used: 4.0 GB


In [8]:
# Cell 4: Prompt builder (mirrors backend/app/generation/llm.py)

def build_prompt(query: str, context: str, language: str, context_found: bool) -> str:
    lang_name = "Bahasa Melayu" if language == "ms" else "English"

    if context_found and context:
        return (
            f"You are a helpful medical assistant. "
            f"Answer using ONLY this context. Cite as [Source N: Title]. "
            f"Answer in {lang_name}.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {query}"
        )
    else:
        return (
            f"You are a helpful medical assistant. "
            f"Answer using medical knowledge. No citations needed. "
            f"Answer in {lang_name}.\n\n"
            f"Question: {query}"
        )

print("Prompt builder ready")

Prompt builder ready


In [ ]:
# Cell 5: Generation function

def generate(query: str, context: str, language: str, context_found: bool) -> str:
    prompt = build_prompt(query, context, language, context_found)

    messages = [
        {"role": "user", "content": prompt}
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,   # was 768 — lower = much faster (was causing 15–38s delays)
            do_sample=False,      # greedy decoding, fastest
            temperature=None,     # must be None when do_sample=False
            top_p=None,           # must be None when do_sample=False
        )

    # Decode only the new tokens (skip the input)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer

# Quick test
test_answer = generate("What is diabetes?", "", "en", False)
print("Test answer:", test_answer[:300], "...")

In [ ]:
# Cell 6: Flask server + ngrok tunnel
import threading
import time
from flask import Flask, request, jsonify
from pyngrok import ngrok

flask_app = Flask(__name__)

@flask_app.route("/generate", methods=["POST"])
def generate_endpoint():
    data = request.get_json()
    try:
        answer = generate(
            query=data["query"],
            context=data.get("context", ""),
            language=data.get("language", "en"),
            context_found=data.get("context_found", False),
        )
        return jsonify({"answer": answer})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@flask_app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "healthy", "model": MODEL_ID})

@flask_app.route("/warmup", methods=["GET"])
def warmup():
    # Called every 4 min by backend keepalive to prevent Kaggle idle
    return jsonify({"status": "warm"})

# Start Flask in background thread
# threaded=True allows Flask to handle multiple requests concurrently
threading.Thread(
    target=lambda: flask_app.run(host="0.0.0.0", port=5000, use_reloader=False, threaded=True),
    daemon=True,
).start()
time.sleep(2)  # wait for Flask to start

# Start ngrok tunnel
ngrok.set_auth_token(secret_value_1)
tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print("=" * 60)
print(f"MEDGEMMA_URL = {public_url}")
print("=" * 60)
print("Copy the URL above into your backend .env file.")
print("Kaggle sessions last up to 12 hrs and survive tab close.")

In [ ]:
# Cell 7: Test the endpoint locally
import requests

resp = requests.post(
    f"{public_url}/generate",
    json={
        "query": "Apakah ubat kencing manis?",
        "context": "",
        "language": "ms",
        "context_found": False,
    },
)
print(f"Status: {resp.status_code}")
print(f"Answer: {resp.json()['answer'][:300]}...")

In [ ]:
# Cell 8: Keep-alive — runs in background thread (does NOT block the kernel)
import requests
import threading
import time

def _keepalive_loop():
    while True:
        time.sleep(300)  # every 5 minutes
        try:
            r = requests.get(f"{public_url}/health", timeout=10)
            print(f"[{time.strftime('%H:%M:%S')}] Keepalive: {r.status_code}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M:%S')}] Keepalive failed: {e}")

threading.Thread(target=_keepalive_loop, daemon=True).start()
print("Keep-alive thread started (pings every 5 min in background). You can continue using other cells.")